## Drift detection
Using another heavy autoencoder model can not be ideal for limited HW scenario.
Try to find out how to execute a less expensive drift detection operation before developing the microservice for drift detection into the 5G k8s Layer.
---------------------------------------------------------------------------

> I want use a geometric apprach.

----------------------------------------------------------------------------
This project has been devided around two main stages:
1. load the CNN and the dataset, execute the model and extract all the embeddings
2. try to check the embedding distribution and see some patterns usefull for the goal

----------------------------------------------------------------------------
## WHAT I AM INTERESTED ON
 1. Images with particular shapes beacuse otherwise the generalization property of the model can extract a good emebdding as well

 2. How model manage images coming outside the Grocery domain
---------------------------------------------------------------------------

reload all the dataset is too expensive, start form the stage 2 load the embedding and plot all the distributions

## ASSUMPTION

1. Please consider that my model is not enterprise, on test set get 85% of accuracy, some rare classes are a issue because original dataset have unbalance distribution and not enough data for hard classification task

 2. I'm developing a best-effort approach for a real-time microservice that performs drift detection filtering to be deploy into the 5G layer

## STAGE 1

In [ ]:
!pip install image-classifiers
!git clone https://github.com/marcusklasson/GroceryStoreDataset.git # training dataset

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from huggingface_hub import hf_hub_download
import tensorflow as tf
import keras
from keras import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from classification_models.tfkeras import Classifiers
from google.colab import drive
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import plotly.express as px
import seaborn as sns
from sklearn.neighbors import NearestNeighbors

from huggingface_hub import snapshot_download

Load the my CNN model for GroceryStore detection, accuracy on the test set (runtime) of 85%

Grocery Store detection -> is my CNN model for testing, already developed by me. Testing is independent from the single cnn and more general

In [ ]:
REPO_ID = "tomasconti/Drift_Detection"
print("Download RestNet18 model")

config_path = hf_hub_download(repo_id=REPO_ID, filename="resnet18_v1/config.json")
weights_path = hf_hub_download(repo_id=REPO_ID, filename="resnet18_v1/model.weights.h5")

with open(config_path, "r") as f:
    model_config = json.load(f)

model = keras.saving.deserialize_keras_object(model_config)
model.load_weights(weights_path)
print("\nRestNet18 loaded")
model.summary()

ResNet18, preprocess_input = Classifiers.get('resnet18') # need of preprocess_input for the ImageDataGenerator, creating streaming sample batch
embedding_extractor = Model(inputs=model.input, outputs=model.get_layer("embedding_layer").output)


In [ ]:
ROOT_DIR = "/content/GroceryStoreDataset/dataset/"
def process_dataframe(df, root_dir):
    df['path'] = df['path'].apply(lambda x: os.path.join(root_dir, x))
    df['fine_label'] = df['fine_label'].astype(str)
    return df

train_df = pd.read_csv(os.path.join(ROOT_DIR, "train.txt"), header=None, sep=",", names=['path', 'fine_label', 'coarse_label'])
val_df = pd.read_csv(os.path.join(ROOT_DIR, "val.txt"), header=None, sep=",",names=['path', 'fine_label', 'coarse_label'])
test_df = pd.read_csv(os.path.join(ROOT_DIR, "test.txt"), header=None, sep=",", names=['path', 'fine_label', 'coarse_label'])

for df in (train_df, val_df, test_df):
    df.drop(columns=["coarse_label"], inplace=True)

train_df = process_dataframe(train_df, ROOT_DIR)
val_df = process_dataframe(val_df, ROOT_DIR)
test_df = process_dataframe(test_df, ROOT_DIR)

combined_df = pd.concat([train_df, val_df], ignore_index=True) #This operation has been explained into the cnn model dev. I need beacause valset does not have all the classes

train_new, val_new = train_test_split(
    combined_df,
    test_size=0.20,
    stratify=combined_df['fine_label'],
    random_state=42
)

print(f"Train: {len(train_new)} | Val: {len(val_new)} | Test: {len(test_df)}")
print(f"Number of fine-grained classes: {combined_df['fine_label'].nunique()}\n")

print( len(train_df['fine_label'].unique()))
print(train_df['fine_label'].unique())

print( len(val_df['fine_label'].unique()))
print(val_df['fine_label'].unique())

print( len(test_df['fine_label'].unique()))
print(test_df['fine_label'].unique())

print( len(train_new['fine_label'].unique()))
print( len(val_new['fine_label'].unique()))
print( len(test_df['fine_label'].unique()))

train_new.head(5)


In [ ]:
def create_generator_from_dataframe(df, batch_size=32):
    datagen = ImageDataGenerator(preprocessing_function=preprocess_input) #ideal to define data streaming flow to get into the models
    # 256 x256 becasue the model first layer already resize the imagine into 224 an 224
    #lazily-loaded data stream
    return datagen.flow_from_dataframe( dataframe=df, x_col='path', y_col='fine_label', target_size=(256, 256), batch_size=64, class_mode='categorical', shuffle=False, validate_filenames=False )
train_gen = create_generator_from_dataframe(train_new)
val_gen   = create_generator_from_dataframe(val_new)
test_gen  = create_generator_from_dataframe(test_df)

Use those cell if i want loading from the local dir instead hugging

In [ ]:

"""
YOGURT_PATH = "/content/drive/MyDrive/Yogurt "
STILL_WATER_PATH = "/content/drive/MyDrive/Still_water_images"
def plot_random_images_inline(folder_path, title_prefix):

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg'))] # extract all the valids file name into the dir path

    num_to_sample = min(3, len(files)) #peack random 3
    random_files = random.sample(files, num_to_sample)

    fig, axes = plt.subplots(1, num_to_sample, figsize=(15, 5))
    for i, img_name in enumerate(random_files):
        img = Image.open(os.path.join(folder_path, img_name)) #open the random file name images and plot it
        img = ImageOps.exif_transpose(img)

        axes[i].imshow(img)
        axes[i].set_title(f"{title_prefix}: {img_name}", fontsize=10)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

plot_random_images_inline(YOGURT_PATH, "Yogurt")
plot_random_images_inline(STILL_WATER_PATH, "Still water")
"""
"""
#preprocess_input -----> allow the preprocessing operation needed for restnest 18 elaboration (example standard scaling ...)
def create_generator_from_dataframe(df, batch_size=32):
    datagen = ImageDataGenerator(preprocessing_function=preprocess_input) #ideal to define data streaming flow to get into the models
    # 256 x256 becasue the model first layer already resize the imagine into 224 an 224
    return datagen.flow_from_dataframe( dataframe=df, x_col='path', y_col='fine_label', target_size=(256, 256), batch_size=batch_size, class_mode='categorical', shuffle=False, validate_filenames=False )

def create_generator_from_folder(folder_path, batch_size=32):
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg'))]

    df = pd.DataFrame({'path': [os.path.join(folder_path, f) for f in image_files]}) #extract the absolute path of images

    datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

    return datagen.flow_from_dataframe( dataframe=df, x_col='path', y_col=None, target_size=(256, 256), batch_size=batch_size, class_mode=None, shuffle=False, validate_filenames=False )

train_gen = create_generator_from_dataframe(train_new)
val_gen   = create_generator_from_dataframe(val_new)
test_gen  = create_generator_from_dataframe(test_df)

yogurt_gen = create_generator_from_folder(YOGURT_PATH)
water_gen  = create_generator_from_folder(STILL_WATER_PATH)
"""

In [ ]:
#laod the hugging face repo into LOCAL_REPO
#/root/.cache/huggingface/hub/models--tomasconti--Drift_Detection/snapshots/1ec8152c78669....
REPO_ID = "tomasconti/Drift_Detection"
LOCAL_REPO = snapshot_download( repo_id=REPO_ID, repo_type="model" )
print("Repo loaded into :", LOCAL_REPO)

def get_local_files(folder_keyword):
    image_files = []
    keyword = folder_keyword.lower()

    for root, _, filenames in os.walk(LOCAL_REPO): #recursion looking for the dir with all that that start by keyward
        if keyword in root.lower():
            #print(f"\n root fine out ->  {root.lower()}")
            #print(f"\n file system fine out ->  {filenames}")
            for filename in filenames: #for each file
                if filename.lower().endswith((".png", ".jpg")): #check if it is a images
                    image_files.append(os.path.join(root, filename)) #laod it

    return sorted(image_files)


def create_generator_from_repo_v2(folder_name, batch_size=32):
    file_paths = get_local_files(folder_name) #go into the folder

    if not file_paths:
        raise ValueError(
            f" ERROR no images'{folder_name}'."
        )

    print(f"\n {folder_name}: with inside {len(file_paths)} imagies")

    df = pd.DataFrame({ "path": file_paths  })

    datagen = ImageDataGenerator( preprocessing_function=preprocess_input )

    return datagen.flow_from_dataframe(
        dataframe=df,
        x_col="path",
        y_col=None,
        target_size=(256, 256),
        batch_size=64,
        class_mode=None,
        shuffle=False,
        validate_filenames=False
    )


for cls in ["Still_water_images", "Yogurt", "Eags"]:
    print(cls, len(get_local_files(cls)))
#return the DirectoryIterator class for each set, lazily-loaded datastream
water_gen = create_generator_from_repo_v2("Still_water_images")
yogurt_gen = create_generator_from_repo_v2("Yogurt")
eags_gen = create_generator_from_repo_v2("Eags")

A very general plot, see different samples


Plot some drift images
Important to consider different position and setting of item
The eggs cover sometimes is close (very important)

In [ ]:
def plot_grid_from_generators(generators_dict, num_samples):

    categories = list(generators_dict.keys())
    n_cats = len(categories)
    fig, axes = plt.subplots(n_cats, num_samples, figsize=(num_samples * 4, n_cats * 4))

    if n_cats == 1 and num_samples == 1: axes = [[axes]]
    elif n_cats == 1: axes = [axes]
    elif num_samples == 1: axes = [[ax] for ax in axes]

    for row, cat in enumerate(categories):
        gen = generators_dict[cat]

        df_files = gen.filenames


        actual_samples = min(num_samples, len(df_files))
        sample_files = random.sample(df_files, actual_samples)

        for col, file_path in enumerate(sample_files):
            img = Image.open(file_path)
            img = ImageOps.exif_transpose(img)

            axes[row][col].imshow(img)
            axes[row][col].set_title(f"{cat}", fontsize=12)
            axes[row][col].axis('off')

        for col in range(actual_samples, num_samples):
            axes[row][col].axis('off')

    plt.tight_layout()
    plt.show()
#dictionary
my_generators = {  "Still Water": water_gen, "Yogurt": yogurt_gen, "Eags": eags_gen }
plot_grid_from_generators(my_generators, num_samples=10)

In [ ]:
my_generators2 = {  "Train": train_gen}
plot_grid_from_generators(my_generators2, num_samples=10)

Load all images and extract embeddings from the model's final layer.

Embeddings are vector representations of items in the model's latent feature space.

In [ ]:
def extract_and_save(generator, name):
    generator.reset()
    embeddings = embedding_extractor.predict(generator, verbose=1) #model get in is a image streaming
    np.save(f"/content/{name}_embeddings.npy", embeddings)
    print(f"Saved {name}_embeddings.npy and hape: {embeddings.shape}")
#lazily-loaded stream as input to models
extract_and_save(train_gen, "grocery_train")
extract_and_save(val_gen,   "grocery_val")
extract_and_save(test_gen,  "grocery_test")
extract_and_save(yogurt_gen, "yogurt")
extract_and_save(water_gen,  "still_water")

## STAGE 2
Starting loading the dataset from the hugging face repo. it is also possible loading form the google colab session

Those embedding come from the stage1

In [ ]:
REPO_ID = "tomasconti/Drift_Detection"
print("📥 Downloading native 512-D embeddings from HuggingFace Hub...")

path_train = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/grocery_train_embeddings.npy")

path_yogurt = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/yogurt_embeddings.npy")

path_test = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/grocery_test_embeddings.npy")
path_val = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/grocery_val_embeddings.npy")
path_water = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/still_water_embeddings.npy")
path_egg = hf_hub_download(repo_id=REPO_ID, filename="good_to_delivery/egg_embeddings.npy")

test_embeddings = np.load(path_test)
val_embeddings = np.load(path_val)
train_embeddings = np.load(path_train)
embeddings_water = np.load(path_water)
yogurt_embeddings = np.load(path_yogurt)
egg_embeddings = np.load(path_egg)

In [ ]:
def explore_embeddings(data, label):
    print(f"\n--- Analysis: {label} ---")
    print(f"Shape: {data.shape}")
    print(f"Mean value: {np.mean(data):.4f}")
    print(f"Standard Deviation: {np.std(data):.4f}")
    print(f"Range: [{np.min(data):.4f}, {np.max(data):.4f}]")

    # Are there any NaNs or Infinities? cloud be a problem in case
    has_nan = np.isnan(data).any()
    has_inf = np.isinf(data).any()
    print(f"Contains NaN: {has_nan} | Contains Inf: {has_inf}")

explore_embeddings(train_embeddings, "Grocery Train")
explore_embeddings(yogurt_embeddings, "Yogurt")
explore_embeddings(test_embeddings, "Grocery Test")
explore_embeddings(val_embeddings, "val ")
explore_embeddings(train_embeddings, "train")
explore_embeddings(egg_embeddings, "egg")

## Principles
  1. A CNN extracts hierarchical features through convolutional kernels whose weights are optimized via ADAM to minimize loss.

  2. These weights function as highly specialized feature detectors for the training domain. Embeddings serve as the final output of this pipeline, condensing complex visual data into a meaningful latent vector.

  3. If the input data introduces novel features, the model's representation drifts, causing the resulting embeddings to deviate from the expected semantic space.

In [ ]:
data = { #dictionary
    "Train": train_embeddings,
    "Val": val_embeddings,
    "Test": test_embeddings,
    "Yogurt": yogurt_embeddings,
    "Water": embeddings_water,
    "Egg": egg_embeddings
}

In [ ]:

#for each of the set of embedding creare a hist to show up the values distributions

plt.figure(figsize=(12, 7))
for label, embeddings in data.items(): #over all the dataset of embeddings
    plt.hist(embeddings.flatten(), bins=50, alpha=0.3, label=label, density=True)

plt.legend()
plt.title("Embedding Distribution Comparison - All Datasets")
plt.xlabel("Feature Value")
plt.ylabel("Density")
plt.grid(axis='y', alpha=0.3)
plt.show()

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

#do not consider all the dataset but one set at time
for i, (label, embeddings) in enumerate(data.items()): #one dataset after the other
    axes[i].hist(embeddings.flatten(), bins=50, color='skyblue', alpha=0.7, density=True)
    axes[i].set_title(f"Distribution: {label}")
    axes[i].set_xlabel("Feature Value")
    axes[i].set_ylabel("Density")
    axes[i].grid(axis='y', alpha=0.3)
#axes[5].axis('off') #empty

plt.tight_layout()
plt.show()

## Embedding Distribution Analysis
#1. Sparse Representation:
    The spike at 0 is primarily due to ReLU activations deactivating neurons
# 2. Domain Alignment:
    The overlapping histograms confirm the model uses a
    consistent feature space for all categories !!!!!!!!!!, ensuring our t-SNE
    visualizations reflect semantic differences !!!!!!!!!!!!!!!

Now try to plot the distribution into a 2D space but also into a 3D optional space to see if there are some pattern usefull for the drift detection operation

In [ ]:
datasets = ["Train", "Val", "Test"] #visualize alone
combined_data = np.concatenate([data["Test"], data["Water"], data["Yogurt"], data["Egg"]]) #create big dataset combine
#creation of labels
combined_labels = ["Test"]*len(data["Test"]) + ["Water"]*len(data["Water"]) + ["Yogurt"]*len(data["Yogurt"]) +["Egg"]*len(data["Egg"])

def run_tsne(embeddings, n_dim):
    #perplexity -> check more neighbornhood, more stability | pca -> algorithm to reduce the dim
    return TSNE(n_components=n_dim, perplexity=50, random_state=42, init="pca").fit_transform(embeddings)

projs_2d = { name: run_tsne(data[name], 2) for name in datasets} # exec tsne on each dataset
projs_2d["Combined"] = run_tsne(combined_data, 2) #exec on all the dataset combine

projs_3d = {name: run_tsne(data[name], 3) for name in datasets}
projs_3d["Combined"] = run_tsne(combined_data, 3)
#################################################################################################
# 2D PLOT
fig_2d, axes_2d = plt.subplots(2, 2, figsize=(14, 12))
axes_2d = axes_2d.flatten()
#single 2d plot
for i, name in enumerate(datasets):
    axes_2d[i].scatter(projs_2d[name][:, 0], projs_2d[name][:, 1], alpha=0.6, s=20)
    axes_2d[i].set_title(f"{name} Set (2D)")
    axes_2d[i].grid(alpha=0.3)
#combine 2d plot
for label in np.unique(combined_labels):
    mask = (np.array(combined_labels) == label)
    axes_2d[3].scatter(projs_2d["Combined"][mask, 0], projs_2d["Combined"][mask, 1], label=label, alpha=0.6, s=30)
axes_2d[3].legend()
axes_2d[3].set_title("Test + Water + Yogurt (2D)")
plt.tight_layout()
plt.show()

# # OPTIONAL THE 3D PLOT
fig_3d = plt.figure(figsize=(14, 12))
for i, name in enumerate(datasets):
    ax = fig_3d.add_subplot(2, 2, i+1, projection='3d')
    ax.scatter(projs_3d[name][:, 0], projs_3d[name][:, 1], projs_3d[name][:, 2], alpha=0.6, s=20)
    ax.set_title(f"{name} Set (3D)")

ax_comb = fig_3d.add_subplot(2, 2, 4, projection='3d')
for label in np.unique(combined_labels):
    mask = (np.array(combined_labels) == label)
    ax_comb.scatter(projs_3d["Combined"][mask, 0], projs_3d["Combined"][mask, 1], projs_3d["Combined"][mask, 2], label=label, alpha=0.6, s=30)
ax_comb.legend()
ax_comb.set_title("Test + Water + Yogurt (3D Static)")
plt.tight_layout()
plt.show()

# OPTIONAL THE 3D DYNAMIC PLOT
df_3d = pd.DataFrame(projs_3d["Combined"], columns=['x', 'y', 'z'])
df_3d['Label'] = combined_labels
fig = px.scatter_3d(df_3d, x='x', y='y', z='z', color='Label', title="Test + Water + Yogurt (3D Interactive)", opacity=0.7)
fig.show()

## Analysis
Drift images embedding are visible, creating distinct blobs into space

Very interesting ---> the egg blue isolated island are probably the eggs photos with cover box !!! differents features!!!!!!!!

Kmean clustering and create the cluster head for each  train distribution,semantic ANCHOR

Drift detection based on eucledian ditance to K closest centroid of the train distribution

# DRIFT DETECTION POLICY -> if an embedding is very far away from the K=1 centroid according to eucleadia distance can be a drifted image

In [ ]:
K_MAX = 1  # Change this variable to set the maximum K value of KNN
data = { #data set of embedding
    "Train": train_embeddings,
    "Val": val_embeddings,
    "Test": test_embeddings,
    "Yogurt": yogurt_embeddings,
    "Water": embeddings_water,
    "Egg": egg_embeddings,
}
#train a Kmean only on train set, can be tested also DBscan and AgglomerativeClustering!!!!!!!!!!!!!!!!!!
# DBscan -> i do not know because some algorithms have some issue with cluster that show up differents densiti into data
kmeans = KMeans(n_clusters=81, n_init=10, random_state=42).fit(data["Train"])
centroids = kmeans.cluster_centers_ #get the cluster centorid
# exec tsne on all all_data to allow comparison, need the same distribution reduction to all
all_data = np.concatenate([*data.values(), centroids])
all_2d = TSNE(n_components=2, perplexity=50, random_state=42, init="pca").fit_transform(all_data)
###################################################################################
#plot centroid based on trian and val
projections = {}
idx = 0
for name, emb in data.items():
    projections[name] = all_2d[idx:idx + len(emb)]
    idx += len(emb)
centroids_2d = all_2d[idx:]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, name in enumerate(["Train", "Val"]):
    axes[i].scatter(projections[name][:, 0], projections[name][:, 1], s=15, alpha=0.4, label=f"{name} Data")
    axes[i].scatter(centroids_2d[:, 0], centroids_2d[:, 1], marker="X", s=100, c="red", edgecolors="black", label="Centroids")
    axes[i].set_title(f"{name} Set & Centroids"); axes[i].legend()
##################################################################################

In [ ]:
knn = NearestNeighbors(n_neighbors=1, metric='euclidean').fit(centroids)
print(knn.metric)
#from sklearn.neighbors import NearestNeighbors
#knn = NearestNeighbors(n_neighbors=1, metric='euclidean')
#knn = NearestNeighbors(n_neighbors=1, metric='cosine')
distances = {}
for name, emb in data.items(): # ex ("train", [3, 4, 5, ...])
    d, _ = knn.kneighbors(emb) #based on centroids -> by default use Minkowski with p=2 as eucleadian distanse =====> runtime complexity O(d*C) number of vector dimension 512 and number of classes 81 but not fixed
    distances[name] = d.flatten()   # 1D array
# Now for each data set collect a distance distribution to K=1 closest centroid
#define a th for drift filtering
val_dists = distances["Val"]
q1, q3 = np.percentile(val_dists, [25, 75])
iqr = q3 - q1
threshold = q3 + 1.5 * iqr
print(f"Threshold computed on Val set: {threshold:.3f}")
############################################################################################
# plot distrances G. like distribution
plt.figure(figsize=(10, 6))
colors = sns.color_palette("viridis", n_colors=len(data))

for i, (label, dist) in enumerate(distances.items()):
    sns.kdeplot(dist, fill=True, alpha=0.2, color=colors[i], label=label)
    # vertical line at the median
    plt.axvline(np.median(dist), color=colors[i], linestyle="--", linewidth=1)

plt.axvline(threshold, color='red', linewidth=2, label=f"Threshold = {threshold:.2f}")
plt.title("Distance to nearest centroid (K=1) – Density plot")
plt.xlabel("Euclidean distance")
plt.ylabel("Density")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()
##########################################################################################
#distences box plots for distribution
rows = []
for name, dist_array in distances.items():
    for d in dist_array:
        rows.append({"Dataset": name, "Distance": d})
df = pd.DataFrame(rows)

plt.figure(figsize=(12, 5))
sns.boxplot(x="Dataset", y="Distance", data=df, palette="viridis", hue="Dataset")
plt.axhline(threshold, color='red', linestyle='--', label=f"Threshold = {threshold:.2f}")
plt.title("Distance distribution (K=1) – Boxplot")
plt.legend()
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.show()

## Analysis
Result are very close to autoencoder approach, but in this case no gpu.

Egg distribution coverage box (i think) can be close to some training class, maybe to milk box of the model domain.
But the idea still working well.

## REMEMBER, BEST EFFORT approach
-------------------------------------------------------------------------------
# HOW TO IMPLEMENT THE MICROSERVICE NOW ?
The idea is using the KNN model loaded and manage as a kserve model(as CNN).
But also include a REDIS db able to execute real time window analysis based on the streaming result of the drift.